# g-and-k: the general NSM-Bayes method (normalising flow)

**The method in one line.** A normalising flow $q_\phi(x\mid\theta)$ is trained as a
neural likelihood on simulations. Instead of plugging it into Bayes' rule directly
(which is fragile under misspecification), we use its *score* in a weighted
score-matching loss and form the generalised posterior

$$
p_\beta(\theta\mid x_{1:n}) \;\propto\; \pi(\theta)\,\exp\!\big\{-\beta\, n\, L_{\mathrm{NSM}}(\theta; x_{1:n})\big\},
$$

$$
L_{\mathrm{NSM}}(\theta) = \frac1n\sum_{i=1}^n \Big[w^2(x_i)\,\big\|\nabla_x \log q_\phi(x_i\mid\theta)\big\|^2
+ 2\,\nabla_x w^2(x_i)^\top \nabla_x \log q_\phi(x_i\mid\theta)
+ 2\,w^2(x_i)\,\Delta_x \log q_\phi(x_i\mid\theta)\Big].
$$

The weight $w$ (inverse multiquadric) down-weights observations far from the bulk
of the data, and the temperature $\beta$ is calibrated by bootstrap so that
$(1-\alpha)$ credible sets have $(1-\alpha)$ coverage (GPC).

**Steps**
1. Seed and build the simulator / prior
2. Simulate training data
3. Simulate observed data and add outliers
4. Train the normalising-flow likelihood $q_\phi$
5. Baseline: the standard NLE posterior on the contaminated data
6. Build the generalised (score-matching) posterior
7. Draw base samples at $\beta_{\text{base}}$
8. Calibrate $\beta$ with GPC
9. Sample the calibrated posterior and compare

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from nsm_bayes.utils import set_seed
from nsm_bayes.simulators.benchmark_simulators.simulators import GandKSimulator
from nsm_bayes.simulators.add_noise import add_outliers_by_proportion
from nsm_bayes.models.general import GeneralModel
from nsm_bayes.models.general.mcmc_settings import MCMCSettings
from nsm_bayes.models.general.posterior import GeneralPosterior
from nsm_bayes.inference.general.calibrate import calibrate_beta
from nsm_bayes.utils.weights import imq_weight

## 1. Seed, simulator and prior

`run_experiment` seeds every repetition with `seed_offset + rep`; for g-and-k
`seed_offset = 1`, so repetition 0 uses seed 1. All data are generated before any
training, in the same order as `experiments/gnk.generate_data`, so the data below
are identical to what the script writes for repetition 0.

The g-and-k distribution is defined through its quantile function
$x = A + B\,(1 + 0.8\tanh(gz/2))\,(1+z^2)^k\,z$ with $z\sim\mathcal N(0,1)$.
The simulator works with $\theta = (A, \log B, g, \log k)$, and the Gaussian prior
is placed on that vector. Each observation is a scalar, so $d_x = 1$.

In [ ]:
set_seed(1)  # seed_offset (1) + repetition index (0)

sim = GandKSimulator(
    prior_mean=torch.tensor([0.0, 0.7, 0.0, -1.5]),          # prior_mean
    prior_cov=torch.tensor([[5.0, 0.0, 0.0, 0.0],            # prior_cov
                            [0.0, 0.5, 0.0, 0.0],
                            [0.0, 0.0, 4.0, 0.0],
                            [0.0, 0.0, 0.0, 0.25]]),
)
prior = sim.prior
theta_true = torch.tensor([1.0, 0.5, 1.0, -1.0])             # theta_true

labels = ["A", "log B", "g", "log k"]
print("prior mean:", prior.loc)
print("prior std: ", prior.stddev)
print("theta_true:", theta_true)

## 2. Training data

Draw parameters from the prior and simulate one observation for each, giving
`num_samples` pairs $(\theta_j, x_j)$ to train the likelihood estimator on.

In [ ]:
theta_sim = prior.sample((100_000,))              # num_samples
x_sim = sim.simulate(theta_sim).unsqueeze(-1)     # (num_samples, d_x = 1)

print("theta_sim:", tuple(theta_sim.shape), " x_sim:", tuple(x_sim.shape))

In [ ]:
# g-and-k has very heavy tails, so show the central 98% of the training data
lo, hi = np.percentile(x_sim.numpy(), [1, 99])
plt.figure(figsize=(6, 3.5))
plt.hist(x_sim.numpy().ravel(), bins=100, range=(lo, hi), color="0.6")
plt.xlabel("x"); plt.ylabel("count")
plt.title("Training simulations (central 98%)")
plt.show()

## 3. Observed data with outliers

`n_obs` observations are drawn at $\theta^\star$. To create misspecification, a
fraction `epsilon` of them is shifted by a value from `outlier_values`
(`add_outliers_by_proportion` *adds* the value, so here 10 points move by $-50$).

In [ ]:
x_obs = sim.simulate(theta_true, n=100).unsqueeze(-1)    # n_obs -> (100, 1)
x_obs_mis, outlier_indices = add_outliers_by_proportion(
    x_obs,
    0.1,       # epsilon
    [-50.0],   # outlier_values
)

print(f"{len(outlier_indices)} of {x_obs.shape[0]} observations replaced by outliers")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(x_obs.numpy().ravel(), bins=30, color="0.6")
axes[0].set_title("Clean observations")
is_out = torch.zeros(x_obs.shape[0], dtype=torch.bool)
is_out[outlier_indices] = True
axes[1].hist(x_obs_mis[~is_out].numpy().ravel(), bins=30, color="0.6", label="inliers")
axes[1].hist(x_obs_mis[is_out].numpy().ravel(), bins=10, color="C3", label="outliers")
axes[1].set_title("Contaminated observations (what we condition on)")
axes[1].legend()
for ax in axes:
    ax.set_xlabel("x")
plt.tight_layout(); plt.show()

## 4. Train the normalising-flow likelihood $q_\phi(x\mid\theta)$

`GeneralModel` wraps an `sbi` `SNLE` trainer. `gnk.yaml` does not set
`density_estimator`, so the default masked autoregressive flow (`"maf"`) is used.
The `MCMCSettings` hold the sampler configuration used both for the NLE baseline
and for the slice sampler of the generalised posterior.

In [ ]:
mcmc = MCMCSettings(
    num_samples=500,   # num_posterior_samples
    num_chains=1,      # num_chains
    warmup_steps=500,  # warmup_steps
    thin=1,            # thin
)

flow_model = GeneralModel(prior, mcmc_settings=mcmc, density_estimator="maf")
flow_model.fit(theta_sim, x_sim)

A quick sanity check: the learned likelihood at $\theta^\star$ should match the
histogram of clean observations.
Note that with $d_x = 1$ sbi's MAF reduces to a conditional Gaussian (sbi warns
about this during training), so $q_\phi(\cdot\mid\theta)$ cannot capture the skew
and heavy tails of the g-and-k exactly.

In [ ]:
x_grid = torch.linspace(-4, 10, 400).unsqueeze(-1)
with torch.no_grad():
    log_q = flow_model.estimator.log_prob(x_grid.unsqueeze(0),
                                          condition=theta_true.unsqueeze(0)).squeeze(0)

plt.figure(figsize=(6, 3.5))
plt.hist(x_obs.numpy().ravel(), bins=30, density=True, color="0.8", label="clean x_obs")
plt.plot(x_grid.squeeze(-1), log_q.exp(), "C0", label=r"$q_\phi(x\mid\theta^\star)$")
plt.xlabel("x"); plt.legend(); plt.title("Learned likelihood at the true parameter")
plt.show()

## 5. Baseline: the standard NLE posterior

Plugging the flow straight into Bayes' rule gives
$p(\theta\mid x_{1:n}) \propto \pi(\theta)\prod_i q_\phi(x_i\mid\theta)$, sampled with
`sbi`'s slice sampler. Because every observation, outliers included, enters the
likelihood with full weight, this posterior is pulled away from $\theta^\star$.

In [ ]:
samples_nle_mis = flow_model.sample_nle_posterior(x_obs_mis)
print("NLE posterior samples:", tuple(samples_nle_mis.shape))
print("NLE posterior mean:   ", samples_nle_mis.mean(0))

## 6. The generalised (score-matching) posterior

`GeneralPosterior` bundles the flow, the contaminated data and the prior. On
construction it estimates the location $\hat\mu$ and inverse scale
$\hat\Sigma^{-1}$ of the observed data (`robust_flag: false` → sample mean and
covariance) and evaluates the squared IMQ weight

$$
w^2(x) = \Big(1 + (x-\hat\mu)^\top\hat\Sigma^{-1}(x-\hat\mu)/c\Big)^{-2}, \qquad c = 1,
$$

together with its gradient for every observation. These are fixed for the rest of
the run.

In [ ]:
posterior = GeneralPosterior(
    x_obs=x_obs_mis,
    flow_model=flow_model,
    prior=prior,
    mcmc=mcmc,
    robust=False,        # robust_flag
    weight_type="imq",   # as in run_normflow_methods
    c=1.0,
)

print("mu_hat:   ", posterior.mu_hat)
print("Sigma_inv:", posterior.Sigma_inv)

In [ ]:
# How the IMQ weight treats each observation
x_line = torch.linspace(-60, 15, 500).unsqueeze(-1)
w2_line, _ = imq_weight(x_line, posterior.mu_hat, posterior.Sigma_inv, c=1.0)

plt.figure(figsize=(7, 3.5))
plt.plot(x_line.squeeze(-1), w2_line, "0.3", lw=1, label=r"$w^2(x)$")
plt.scatter(x_obs_mis[~is_out].squeeze(-1), posterior.w2[~is_out], s=14, label="inliers")
plt.scatter(x_obs_mis[is_out].squeeze(-1), posterior.w2[is_out], s=14, c="C3", label="outliers")
plt.xlabel("x"); plt.ylabel(r"$w^2(x)$"); plt.legend()
plt.title("Squared IMQ weights of the observations")
plt.show()

$L_{\mathrm{NSM}}(\theta)$ can be evaluated directly. It is small near parameters
that explain the bulk of the data well:

In [ ]:
with torch.no_grad():
    print(f"L_NSM(theta_true)  = {posterior.sm_loss(theta_true).item():.4f}")
    print(f"L_NSM(prior mean)  = {posterior.sm_loss(prior.loc).item():.4f}")
    print(f"L_NSM(NLE mean)    = {posterior.sm_loss(samples_nle_mis.mean(0)).item():.4f}")

## 7. Base samples at $\beta_{\text{base}}$

Calibration starts from slice-sampler draws of $p_\beta$ at
`beta_base_general = 1.0`. `posterior.sample(beta)` runs the slice sampler on
$\log\pi(\theta) - \beta\, n\, L_{\mathrm{NSM}}(\theta)$.

In [ ]:
theta_samples_base = posterior.sample(1.0)   # beta_base_general
print("base samples:", tuple(theta_samples_base.shape))
print("base mean:   ", theta_samples_base.mean(0))

## 8. Calibrate $\beta$ (general posterior calibration)

`calibrate_beta` runs `T` stochastic-approximation steps on $\log\beta$. At each
step it

1. draws `B` multinomial bootstrap resamples of the observations,
2. importance-reweights the base samples to approximate the posterior at the
   current $\beta$ for every resample (using the cached per-observation losses),
3. checks how often the $(1-\alpha)$ weighted credible ellipsoid contains
   $\hat\theta = \arg\min_\theta L_{\mathrm{NSM}}(\theta)$, and
4. moves $\log\beta$ by $k_t\,(\widehat{\text{coverage}} - (1-\alpha))$ with
   $k_t = 10/(t+10)$.

If the importance weights degenerate (mean ESS below 30% of the samples) the
sampler is re-run at the current $\beta$. `gnk.yaml` has no `beta_min_general`,
so the driver's fallback `beta_min = 0.01` is used.

In [ ]:
beta, history = calibrate_beta(
    posterior,
    theta_samples_base,
    beta_base=1.0,   # beta_base_general
    alpha=0.05,      # alpha
    B=100,           # B
    T=20,            # T
    beta_min=0.01,   # beta_min_general (not set in gnk.yaml -> driver default)
)
print(f"calibrated beta = {beta:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history["betas"], marker="o")
axes[0].set_yscale("log"); axes[0].set_xlabel("iteration"); axes[0].set_ylabel(r"$\beta$")
axes[0].set_title(r"$\beta$ trajectory")
axes[1].plot(history["coverages"], marker="o")
axes[1].axhline(1 - 0.05, color="C3", ls="--", label=r"target $1-\alpha$")
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("bootstrap coverage")
axes[1].set_ylim(0, 1.05); axes[1].legend()
axes[1].set_title("Estimated coverage")
plt.tight_layout(); plt.show()

## 9. Calibrated posterior and comparison

Finally, sample $p_\beta$ at the calibrated $\beta$ and compare it with the NLE
baseline and the true parameter.

In [ ]:
samples_sm = posterior.sample(beta)

mean_sm = samples_sm.mean(0)
cov_sm = torch.cov(samples_sm.T)
mean_nle = samples_nle_mis.mean(0)

print(f"{'':8s}{'true':>9s}{'NLE':>9s}{'NSM':>9s}{'NSM sd':>9s}")
for k, name in enumerate(labels):
    print(f"{name:8s}{theta_true[k].item():9.3f}{mean_nle[k].item():9.3f}"
          f"{mean_sm[k].item():9.3f}{cov_sm[k, k].sqrt().item():9.3f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for k, ax in enumerate(axes):
    both = torch.cat([samples_sm[:, k], samples_nle_mis[:, k]]).numpy()
    rng = (min(both.min(), theta_true[k].item()), max(both.max(), theta_true[k].item()))
    ax.hist(samples_nle_mis[:, k].numpy(), bins=30, range=rng, density=True,
            alpha=0.5, color="0.5", label="NLE")
    ax.hist(samples_sm[:, k].numpy(), bins=30, range=rng, density=True,
            alpha=0.7, color="#C0504D", label="NSM-Bayes")
    ax.axvline(theta_true[k].item(), color="k", ls="--", lw=1.5, label=r"true $\theta$")
    ax.set_xlabel(labels[k]); ax.set_yticks([])
axes[0].legend(fontsize=8)
fig.suptitle(r"Posterior marginals on contaminated g-and-k data ($\epsilon = 0.1$)")
plt.tight_layout(); plt.show()

### What the script saves

For reference, `run_normflow_methods` writes (per repetition) the flow
(`likelihood_estimator_full`), `samples_nle_mis`, `samples_sm`, `beta`,
`gpc_history`, the posterior moments and timing files. The objects above with the
same names are what it would save for repetition 0.